# Train YOLOv8 known-defect detector on Kaggle (NEU-DET)

Self-contained — no repo cloning, no credentials. Before running:

1. **Add data**: on the right sidebar, click *Add Input* and attach the
   [NEU Surface Defect Database](https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database)
   dataset (same one used locally).
2. **Settings > Accelerator**: set to a GPU (T4 x2 or P100).
3. **Settings > Internet**: ON (needed for `pip install ultralytics`).

Then run all cells top to bottom.

## 1. Locate the attached dataset

Kaggle's exact mount path/nesting can vary by how the dataset was
packaged, so this inspects `/kaggle/input` first rather than assuming
a hardcoded path.

In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    depth = root[len('/kaggle/input'):].count(os.sep)
    if depth >= 3:
        dirs[:] = []
        continue
    print(root)


In [ ]:
import glob

# Auto-detect the NEU-DET "train" split folder (contains images/ and annotations/)
# under whatever nesting Kaggle mounted the dataset at.
candidates = [
    c for c in glob.glob('/kaggle/input/**/train', recursive=True)
    if os.path.isdir(os.path.join(c, 'images')) and os.path.isdir(os.path.join(c, 'annotations'))
]
assert candidates, (
    "Could not auto-locate the NEU-DET train/ folder under /kaggle/input — "
    "check the directory listing printed above and set NEU_DET_ROOT manually."
)
NEU_DET_ROOT = os.path.dirname(candidates[0])
print('Detected NEU-DET root:', NEU_DET_ROOT)


## 2. Install ultralytics

In [ ]:
!pip install -q ultralytics


## 3. Convert PASCAL VOC XML annotations to YOLO format

Same conversion as `preprocessing/voc_to_yolo.py` in the repo, adapted
to Kaggle's writable `/kaggle/working` output directory. The source
dataset ships with no test split, so this holds out 1-in-4 images per
class from validation into test (disjoint, deterministic by filename).

In [ ]:
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

CLASS_NAMES = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

SOURCE_ROOT = Path(NEU_DET_ROOT)
DATASET_ROOT = Path('/kaggle/working/dataset')
TEST_HOLDOUT_EVERY = 4


def convert_annotation(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    width = float(size.findtext('width'))
    height = float(size.findtext('height'))
    lines = []
    for obj in root.findall('object'):
        name = obj.findtext('name')
        if name not in CLASS_TO_ID:
            print(f'  skip unknown class {name!r} in {xml_path.name}')
            continue
        box = obj.find('bndbox')
        xmin, ymin, xmax, ymax = (float(box.findtext(t)) for t in ('xmin', 'ymin', 'xmax', 'ymax'))
        cx = ((xmin + xmax) / 2) / width
        cy = ((ymin + ymax) / 2) / height
        w = (xmax - xmin) / width
        h = (ymax - ymin) / height
        lines.append(f'{CLASS_TO_ID[name]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return lines


def write_example(image_path, xml_path, images_dir, labels_dir):
    lines = convert_annotation(xml_path)
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(image_path, images_dir / image_path.name)
    (labels_dir / f'{image_path.stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))


def collect_examples(split_dir):
    images_root = split_dir / 'images'
    annotations_root = split_dir / 'annotations'
    examples = []
    for class_dir in sorted(p for p in images_root.iterdir() if p.is_dir()):
        for image_path in sorted(class_dir.glob('*.jpg')):
            xml_path = annotations_root / f'{image_path.stem}.xml'
            if xml_path.exists():
                examples.append((image_path, xml_path))
            else:
                print(f'  skip {image_path.name} — no matching annotation')
    return examples


print('Converting train split...')
train_examples = collect_examples(SOURCE_ROOT / 'train')
for image_path, xml_path in train_examples:
    write_example(image_path, xml_path, DATASET_ROOT / 'train' / 'images', DATASET_ROOT / 'train' / 'labels')
print(f'  {len(train_examples)} train examples')

print(f'Converting validation split (holding out 1-in-{TEST_HOLDOUT_EVERY} per class for test)...')
by_class = {}
for image_path, xml_path in collect_examples(SOURCE_ROOT / 'validation'):
    by_class.setdefault(image_path.parent.name, []).append((image_path, xml_path))

val_count = test_count = 0
for class_name, examples in by_class.items():
    for i, (image_path, xml_path) in enumerate(sorted(examples)):
        if i % TEST_HOLDOUT_EVERY == 0:
            write_example(image_path, xml_path, DATASET_ROOT / 'test' / 'images', DATASET_ROOT / 'test' / 'labels')
            test_count += 1
        else:
            write_example(image_path, xml_path, DATASET_ROOT / 'validation' / 'images', DATASET_ROOT / 'validation' / 'labels')
            val_count += 1
print(f'  {val_count} validation examples, {test_count} test examples')


## 4. Write data.yaml

In [ ]:
data_yaml = f"""path: {DATASET_ROOT}
train: train/images
val: validation/images
test: test/images

names:
  0: crazing
  1: inclusion
  2: patches
  3: pitted_surface
  4: rolled-in_scale
  5: scratches
"""
(DATASET_ROOT / 'data.yaml').write_text(data_yaml)
print((DATASET_ROOT / 'data.yaml').read_text())


## 5. Train

`yolov8n.pt` is the nano checkpoint — fastest, good enough to prove the
pipeline. Swap to `yolov8s.pt`/`yolov8m.pt` for better accuracy once this
run works end to end.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data=str(DATASET_ROOT / 'data.yaml'),
    epochs=100,
    imgsz=640,
    batch=16,
    project='/kaggle/working/models/yolov8',
    name='defect_detector',
)


## 6. Evaluate (optional)

In [ ]:
metrics = model.val(
    data=str(DATASET_ROOT / 'data.yaml'),
    split='test',
)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)


## 7. Get the trained weights back to your local repo

`best.pt` lands at `/kaggle/working/models/yolov8/defect_detector/weights/best.pt`.

Zip it below, then use the notebook's **Output** pane (after *Save Version
→ Save & Run All*) to download it. Copy the extracted `best.pt` into your
local repo at `models/yolov8/defect_detector/weights/best.pt` — it's
gitignored, so this is a manual copy, not a git operation.

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/best_weights', 'zip', '/kaggle/working/models/yolov8/defect_detector/weights')
print('Zipped weights at /kaggle/working/best_weights.zip')
